# How would you create the datadframe from nested json and how do you flatten it?

In [39]:
from pyspark.sql import SparkSession

In [40]:
spark=SparkSession.builder.appName('myApp').getOrCreate()

### 1. Reading .json file- Spark automatically infers nested structure.

In [41]:
df=spark.read \
  .option("multiline", 'True') \
  .json("/data.json")

In [43]:
df.printSchema()

root
 |-- customer: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- price: long (nullable = true)
 |    |    |-- product: string (nullable = true)
 |-- order_id: long (nullable = true)



In [44]:
df.show()

+---------+--------------------+--------+
| customer|               items|order_id|
+---------+--------------------+--------+
|{1, John}|[{100, A}, {200, B}]|     101|
| {2, Jia}|[{100, A}, {200, B}]|     102|
+---------+--------------------+--------+



## Using Explicit Schema

In [45]:
from pyspark.sql.types import *

In [46]:
schema=StructType([
    StructField ("order_id",IntegerType(),True),
    StructField ("customer", StructType([
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True)
    ])),
    StructField("items", ArrayType(
        StructType([
          StructField("product", StringType(), True),
          StructField("price", DoubleType(), True)
        ])
    ))
])

In [47]:
df2=spark.read \
  .option('multiline', True) \
  .schema(schema) \
  .json("/data.json")

In [48]:
df2.show(truncate=False)

+--------+---------+------------------------+
|order_id|customer |items                   |
+--------+---------+------------------------+
|101     |{1, John}|[{A, 100.0}, {B, 200.0}]|
|102     |{2, Jia} |[{A, 100.0}, {B, 200.0}]|
+--------+---------+------------------------+



In [55]:
df.select('order_id','customer.id','customer.name','items.product','items.price').show()

+--------+---+----+-------+----------+
|order_id| id|name|product|     price|
+--------+---+----+-------+----------+
|     101|  1|John| [A, B]|[100, 200]|
|     102|  2| Jia| [A, B]|[100, 200]|
+--------+---+----+-------+----------+



### Nested .json flattening

In [57]:
from pyspark.sql.functions import explode

df_exploded = df.withColumn("item", explode("items"))

df_exploded.select(
    "order_id",
    "customer.id",
    "customer.name",
    "item.product",
    "item.price"
).show()

+--------+---+----+-------+-----+
|order_id| id|name|product|price|
+--------+---+----+-------+-----+
|     101|  1|John|      A|  100|
|     101|  1|John|      B|  200|
|     102|  2| Jia|      A|  100|
|     102|  2| Jia|      B|  200|
+--------+---+----+-------+-----+

